# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SymbolPamnani/Flyrank-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. Answer

**Task type: Ranking / Scoring**

The goal is to rank content pages by how strongly they should be considered for review or refresh.

This is a ranking problem because the content team needs a prioritized list of pages to investigate first rather than simply assigning every page a yes/no prediction.

A scoring approach can combine multiple signals and produce a priority score for each content item.

In [14]:
import pandas as pd

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Dataset shape:", df.shape)

Rows: 30000
Columns: 44
Dataset shape: (30000, 44)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Answer

The model will produce a **priority score** for each content page.

For evaluation, `trend_direction` can be used to define a proxy label for pages that are currently declining. Pages with `trend_direction == "down"` are treated as the declining class.

This is a defined proxy based on the available dataset rather than an independently observed future outcome.

The model will therefore be used for decision-support: whether the highest-ranked pages contain a larger share of pages associated with the declining trend.

In [15]:
print("Trend direction counts:")
print(df["trend_direction"].value_counts())

df["declining_proxy"] = (
    df["trend_direction"] == "down"
).astype(int)

print("\nDeclining proxy counts:")
print(df["declining_proxy"].value_counts())

print("\nDeclining proxy percentages:")
print(
    df["declining_proxy"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining proxy counts:
declining_proxy
1    16262
0    13738
Name: count, dtype: int64

Declining proxy percentages:
declining_proxy
1    54.2
0    45.8
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Answer

**Primary metric: Precision@50**

Precision@50 measures how many of the top 50 ranked pages are associated with the declining proxy.

For example, a Precision@50 of 0.70 would mean that 35 of the top 50 ranked pages have the declining proxy.

This metric fits the decision because a content team may only have enough time to investigate a limited number of pages first. The ranking should therefore make the top of the queue useful.

In [16]:
def precision_at_k(scores, labels, k=50):
    ranked_indices = scores.sort_values(
        ascending=False
    ).head(k).index

    return labels.loc[ranked_indices].mean()


# Reference score using the magnitude of negative trends
baseline_score = df["trend_pct"].clip(upper=0).abs()

score = precision_at_k(
    baseline_score,
    df["declining_proxy"],
    k=50
)

print(f"Reference Precision@50: {score:.3f}")

Reference Precision@50: 1.000


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. Answer

The unit of analysis is one **pseudonymized content item/page**.

Each row represents one content item with its search-performance measurements and other available attributes.

The ranking will assign a priority score to each content item, producing one ranking score per row.

In [17]:
print("Dataframe shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nOne row represents one content item/page.")
print("Rows:", len(df))

Dataframe shape: (30000, 45)

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,declining_proxy
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1



One row represents one content item/page.
Rows: 30000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Answer

A fixed rule could prioritize pages using one simple signal, such as a large decline in impressions or clicks.

However, content performance contains multiple signals, including impressions, clicks, sessions, average position, CTR, engagement, content age, and content characteristics.

These signals can interact in ways that are difficult to represent with a small number of hand-written if-statements.

ML earns its place if it can combine these signals and improve Precision@50 over a reasonable fixed-rule baseline on unseen data.

In [18]:
candidate_columns = [
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days"
]

available_columns = [
    col for col in candidate_columns
    if col in df.columns
]

print("Candidate signals available in the dataset:")

for col in available_columns:
    print("-", col)

print("\nNumber of candidate signals:", len(available_columns))

Candidate signals available in the dataset:
- impressions_last_30d
- impressions_prev_30d
- clicks_last_30d
- clicks_prev_30d
- sessions_last_30d
- sessions_prev_30d
- avg_position
- ctr
- engagement_rate
- scroll_rate
- content_age_days

Number of candidate signals: 11


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.